In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

In [0]:
df = spark.read.format("parquet").load(f"abfss://bronze@projdatalake.dfs.core.windows.net/customers")
df.display()

In [0]:
df.printSchema()

In [0]:
# Remove last column
df = df.drop('_rescued_data')

df.display()

#### **Transformations**

In [0]:
# Split the domain section from email column and save in another column
df = df.withColumn('domains', split(df.email, '@').getItem(1))
df.display()

In [0]:
# count number of customers per domain and show in descending order
domain_count = df.groupBy('domains').count().orderBy(col('count').desc())
domain_count.display()

In [0]:
# filter customers of gmail.com domain
gmail_df = df.filter(df.domains == 'gmail.com')
gmail_df.display()
time.sleep(5)

# filter customers of hotmail.com domain
hotmail_df = df.filter(df.domains == 'hotmail.com')
hotmail_df.display()
time.sleep(5)

# filter customers of yahoo.com domain
yahoo_df = df.filter(df.domains == 'yahoo.com')
yahoo_df.display()
time.sleep(5)

In [0]:
# create a full name column and drop first name and last name columns
df = df.withColumn('full_name', concat(df.first_name, lit(' '), df.last_name)).drop('first_name', 'last_name')
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@projdatalake.dfs.core.windows.net/customers")

Next create tables on top of our data

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_wspace.silver.customers_silver
USING DELTA
LOCATION 'abfss://silver@projdatalake.dfs.core.windows.net/customers'

In [0]:
%sql
select * from databricks_wspace.silver.customers_silver